# Metrics Pipeline

Computes the core benchmark metrics over the LLM recommendations: diversity, parity, factuality, consistency, and duplicates.

# Metrics Pipeline

Plots the per-call benchmark metrics over the LLM recommendations: diversity, parity, factuality, consistency, duplicates.

The compute step lives in `code/scripts/metrics/build_valid_calls.py`; this notebook just loads its output and plots.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

sys.path.insert(0, '../..')
%reload_ext autoreload
%autoreload 2

from libs.visuals.vis import sns_paper_style
from libs.visuals.grouped_metrics import plot_grouped_metrics
from libs.visuals.grouped_metrics import plot_grouped_metrics_labeled
from libs.visuals.quadrant_scatter import plot_quadrant_scatter

from libs.metrics.aggregators import aggregate_scores
from libs.metrics.utils import *
from libs.metrics.constants import *
from libs.visuals.constants import *
from libs.utils import ios
from libs.utils.config import get_results_path


In [ ]:
RESULTS = get_results_path()
DATA = RESULTS.parent / 'data'

VALID_CALLS_PATH = RESULTS / 'factualities/tables/valid_requests_metadata.csv'
PLOTS_DIR = RESULTS / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_PROMPT_METADATA_FN = DATA / 'context/english/input.json'
MODELS_METADATA_FN = DATA / 'models/metadata.json'
MODELS_FN = DATA / 'models/models.txt'

MODELS = ios.read_list_from_file(MODELS_FN)
models_metadata = ios.load_json(MODELS_METADATA_FN)
field_metadata = ios.load_json(CONTEXT_PROMPT_METADATA_FN)
SUBFIELD_ORDER = [sf for f in FIELD_ORDER for fobj in field_metadata['fields'] if fobj['field']==f for sf in fobj['subfields']]

FIG_WIDTH = FIG_WIDTH_PAPER
FIG_HEIGHT = FIG_HEIGHT_PAPER
FIG_SIZE = (FIG_WIDTH, FIG_HEIGHT)


# Plots

In [ ]:
df_valid_calls = pd.read_csv(VALID_CALLS_PATH, index_col=False)

def _pct_tier_last(c):
    parts = c.split('_')
    if c.startswith('pct_') and len(parts) == 3 and parts[1] in ('low', 'med', 'high'):
        return f"{parts[0]}_{parts[2]}_{parts[1]}"
    return c
df_valid_calls = df_valid_calls.rename(columns=_pct_tier_last)

print(f'Loaded {len(df_valid_calls):,} rows × {len(df_valid_calls.columns)} columns from {VALID_CALLS_PATH}')

In [ ]:
df_valid_calls.head(2)

In [ ]:
sns_paper_style(font_scale=1.55)

In [ ]:

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
_removed = 0
for _old in PLOTS_DIR.glob('*.pdf'):
    _old.unlink()
    _removed += 1
print(f'Cleared {PLOTS_DIR} — {_removed} stale PDF(s) removed')

## RQ1 & RQ2

In [ ]:
variables = {'persona': PERSONA_VARIABLES, 'context': MAIN_CONTEXT_VARIABLES}

df_metric = pd.DataFrame()
group_configs = []
for i, (prompt_var_class, vars) in enumerate(variables.items()):
    for j, var in enumerate(vars):
        tmp = aggregate_scores(df_valid_calls, [var]).rename(columns={var: 'label'})
        tmp.loc[:,'column'] = var
        df_metric = pd.concat([df_metric, tmp], ignore_index=True)

        group_configs.append({'label': var.split("_en")[0].capitalize(), 
                              'column': 'label', 
                              'color': PROMPT_VAR_COLORS.get(prompt_var_class), 
                              'filter': {'column':var},
                              'order': ORDER_MAP[var],
                              'shade': False})
            
for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_stacked.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_metric,
        group_configs=group_configs,
        row_labels = SENIORITY_MAP | ROLE_MAP | {'Computer Science':'Computer Sc.'},
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(group_configs)/1.1),
        textwrap_width = 20,
        section_style='plain',
        section_label_pad=0.2,
        section_label_align='right',
        row_label_align='left',
        section_label_width=0.5,
        row_label_width=1.9,
        row_label_pad=0.3,
        section_sep_left_pad=-0.22
    )


## RQ3: Insfrastructure

In [ ]:

df_infra = pd.DataFrame()
for c in ['model_access','model_size','model_class']:
    tmp = aggregate_scores(df_valid_calls, [c]).rename(columns={c: 'label'})
    tmp.loc[:,'column'] = c
    df_infra = pd.concat([df_infra, tmp], ignore_index=True)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_infrastructure_{metric_type}.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_infra,
        group_configs=[
            {'label': MODEL_ARCHITECTURE_MAP['model_access'], 'column': 'label', 'filter': {'column': 'model_access'}, 'color': "#ADD59E", 'order': ACCESS_ORDER, 'shade': False},
            {'label': MODEL_ARCHITECTURE_MAP['model_size'],   'column': 'label', 'filter': {'column': 'model_size'},   'color': "#DFD693", 'order': SIZE_ORDER, 'shade': False},
            {'label': MODEL_ARCHITECTURE_MAP['model_class'],  'column': 'label', 'filter': {'column': 'model_class'},  'color': "#D78F93", 'order': REASONING_ORDER, 'shade': False},
        ],
        row_labels=ACCESS_MAP | SIZE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * 2.1),
        textwrap_width = 20,
        section_style='plain',
        section_label_pad=0.2,
        section_label_align='right',
        row_label_align='left',
        section_label_width=0.5,
        row_label_width=1.9,
        row_label_pad=0.3,
        section_sep_left_pad=-0.22
    )


## RQ4: Socio-Technical Trade-off

In [ ]:

df_data_tmp = df_valid_calls.copy()

agg_cols = TECHNICAL_METRICS_NORM + PARITY_METRICS

df_data_tmp.loc[:, 'color'] = df_data_tmp.loc[:,'model_family'].apply(lambda fam: MODEL_FAMILY_COLORS.get(fam, '#6B7280'))

per_model_mean = (df_data_tmp.groupby(['model_short_name','model_family', 'color'], dropna=False, observed=True)[agg_cols]
                  .mean()
                  .reset_index())
per_model_std = (df_data_tmp.groupby(['model_short_name','model_family', 'color'], dropna=False, observed=True)[agg_cols]
                  .std()
                  .reset_index())

per_model_mean['x_technical'] = per_model_mean[TECHNICAL_METRICS_NORM].sum(axis=1, skipna=False)
per_model_mean['y_social']    = per_model_mean[PARITY_METRICS].sum(axis=1, skipna=False)

per_model_std['x_technical'] = per_model_std[TECHNICAL_METRICS_NORM].sum(axis=1, skipna=False)
per_model_std['y_social']    = per_model_std[PARITY_METRICS].sum(axis=1, skipna=False)

print(f'Scatter — {per_model_mean.model_short_name.nunique()} models')
print(f"Technical max: {per_model_mean['x_technical'].max():.3f}, Social max: {per_model_mean['y_social'].max():.3f}")

fig, ax = plot_quadrant_scatter(per_model_mean, label_col="model_short_name", figsize=(FIG_WIDTH, 5.5), show_quadrant_legend=True)

fn = str(PLOTS_DIR / f'plot_technical_vs_social.pdf')
plt.savefig(fn, bbox_inches='tight', dpi=FIG_DPI)
print(fn)

plt.show()
plt.close()


# Additional

___


## K

In [ ]:
df_k = aggregate_scores(df_valid_calls, ['k'])
display(df_k)

for metric_type, metric_list in METRIC_TYPES.items():

    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_k.pdf')
    print(fn)
    plot_grouped_metrics(
        df_k,
        group_configs=[
            {'label': 'Top-k', 'column': 'k', 'color': '#34495E', 'order': K_ORDER},
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=FIG_SIZE
    )

### K and Language

In [ ]:

df_k_lang = aggregate_scores(df_valid_calls, ['k','language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_k_lang.pdf')
    print(fn)
    
    plot_grouped_metrics_labeled(
        df_k_lang,
        group_configs=[
            {'label': str(k), 
             'column': 'language_en', 'color': K_COLORS[k], 'filter': {'k': k}, 'order': LANGUAGE_ORDER} for k in K_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(LANGUAGE_ORDER))
    )


## Language

In [ ]:

df_lang = aggregate_scores(df_valid_calls, ['language_en'])
display(df_lang)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_lang,
        group_configs=[
            {'label': 'Language', 
             'column': 'language_en', 
             'color': '#4A90D9',
             'order': LANGUAGE_ORDER},
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=FIG_SIZE
    )


### Language and K

In [ ]:
df_lang_k = aggregate_scores(df_valid_calls, ['language_en','k'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_lang_k.pdf')
    print(fn)
    
    plot_grouped_metrics_labeled(
        df_lang_k,
        group_configs=[
            {'label': LANGUAGE_MAP[lang], 
             'column': 'k', 
             'color': LANGUAGE_COLORS[lang],
             'filter': {'language_en': lang}, 
             'order': K_ORDER} for lang in LANGUAGE_ORDER
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(LANGUAGE_ORDER))
    )


### Language and Task

In [ ]:

from pydoc import text


df_lang_task = aggregate_scores(df_valid_calls, ['language_en','task_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_task_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_lang_task,
        group_configs=[
            {'label': LANGUAGE_MAP[language], 
             'column': 'task_en', 
             'color': LANGUAGE_COLORS[language],
             'filter': {'language_en': language}, 
             'order': TASK_ORDER}
            for language in LANGUAGE_ORDER
        ],
        row_labels=TASK_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(TASK_ORDER)),
        textwrap_width=25
    )


## Location

In [ ]:

df_loc = aggregate_scores(df_valid_calls, ['location_en'])
display(df_loc)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_loc.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_loc,
        group_configs=[
            {'label': 'Location', 
             'column': 'location_en', 
             'color': '#D95B5B',
             'order': LOCATION_ORDER},
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        row_labels=LOCATION_MAP,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT + (len(LOCATION_ORDER)/8))
    )


### Location and Language

In [ ]:

df_loc_lang = aggregate_scores(df_valid_calls, ['location_en','language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_loc_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_loc_lang,
        group_configs=[
            {'label': LOCATION_MAP[loc], 
             'column': 
             'language_en', 
             'color': LOCATION_COLORS[loc],
             'filter': {'location_en': loc}, 
             'order': LANGUAGE_ORDER} for loc in LOCATION_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(LOCATION_ORDER))   
    );


## Task

In [ ]:
df_task = aggregate_scores(df_valid_calls, ['task_en'])
display(df_task)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_task.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_task,
        group_configs=[
            {'label': 'Task', 
             'column': 'task_en', 
             'color': '#9B59B6',
             'order': TASK_ORDER},
        ],
        row_labels=TASK_MAP_LONG,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * 0.7),
        textwrap_width=20
    )


### Task and K

In [ ]:

df_task_k = aggregate_scores(df_valid_calls, ['task_en','k'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_task_k.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_task_k,
        group_configs=[
            {'label': TASK_MAP_LONG[task_val], 
             'column': 'k', 
             'color': TASK_COLORS[task_val],
             'filter': {'task_en': task_val}, 
             'order': K_ORDER} for task_val in TASK_ORDER
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(TASK_ORDER)),
        textwrap_width=20
    )


## Role

In [ ]:

df_role = aggregate_scores(df_valid_calls, ['role_en'])
display(df_role)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_role.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_role,
        group_configs=[
            {'label': 'Role', 
             'column': 'role_en', 
             'color': '#16A085',
             'order': ROLE_ORDER},
        ],
        row_labels=ROLE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * 0.7)
    )

### Role and Language

In [ ]:

df_role_lang = aggregate_scores(df_valid_calls, ['role_en', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_role_lang.pdf')
    print(fn)
        
    plot_grouped_metrics_labeled(
        df_role_lang,
        group_configs=[
            {'label': ROLE_MAP[role_val], 
             'column': 'language_en', 
             'color': ROLE_COLORS[role_val],
             'filter': {'role_en': role_val}, 
             'order': LANGUAGE_ORDER}
            for role_val in ROLE_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(ROLE_ORDER))
    )

## Field

In [ ]:

df_field = aggregate_scores(df_valid_calls, ['field_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_field.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_field,
        group_configs=[
            {'label': 'Field', 
             'column': 'field_en', 
             'color': '#2ECC71', 
             'order': FIELD_ORDER},
        ],
        row_labels=FIELD_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT + (len(FIELD_ORDER)/5))
    )


### Field and Language

In [ ]:

df_field_lang = aggregate_scores(df_valid_calls, ['field_en', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_field_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_field_lang,
        group_configs=[
            {'label': FIELD_MAP[field_en], 
             'column': 'language_en', 
             'color': FIELD_COLORS[field_en], 
             'filter': {'field_en': field_en}, 
             'order': LANGUAGE_ORDER } for field_en in FIELD_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(FIELD_ORDER))
    )


### Field and K

In [ ]:

df_field_k = aggregate_scores(df_valid_calls, ['field_en', 'k'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_field_k.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_field_k,
        group_configs=[
            {'label': field_en, 
             'column': 'k', 
             'color': FIELD_COLORS[field_en], 
             'filter': {'field_en': field_en}, 
             'order': K_ORDER } for field_en in FIELD_ORDER
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(FIELD_ORDER)),
        textwrap_width=20
    )


## Subfield

In [ ]:

df_subfield = aggregate_scores(df_valid_calls, ['subfield_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_subfield.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_subfield,
        group_configs=[
            {'label': 'Subfield', 
             'column': 'subfield_en', 
             'color': '#2ECC71', 
             'order': SUBFIELD_ORDER},
        ],
        row_labels=SUBFIELD_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT + (len(SUBFIELD_ORDER)/5))
    )


### Subfield and Language

In [ ]:

df_subfield_lang = aggregate_scores(df_valid_calls, ['subfield_en', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_subfield_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_subfield_lang,
        group_configs=[
            {'label': SUBFIELD_MAP[subfield_en], 
             'column': 'language_en', 
             'color': SUBFIELD_COLORS[subfield_en], 
             'filter': {'subfield_en': subfield_en}, 
             'order': LANGUAGE_ORDER } for subfield_en in SUBFIELD_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(SUBFIELD_ORDER))
    )


## Target (seniority)

In [ ]:

df_target = aggregate_scores(df_valid_calls, ['target_en'])
display(df_target)

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_seniority.pdf')
    print(fn)
        
    plot_grouped_metrics_labeled(
        df_target,
        group_configs=[
            {'label': 'Target', 
             'column': 'target_en', 
             'color': '#E8703A',
             'order': SENIORITY_ORDER},
        ],
        row_labels=SENIORITY_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * 0.7)
    )

### Seniority and Language

In [ ]:

df_target_lang = aggregate_scores(df_valid_calls, ['target_en', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_seniority_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_target_lang,
        group_configs=[
            {'label': target_en, 
             'column': 'language_en', 
             'color': SENIORITY_COLORS[target_en],
            'filter': {'target_en': target_en}, 
            'order': LANGUAGE_ORDER}
            for target_en in SENIORITY_ORDER
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(SENIORITY_ORDER)),
        textwrap_width=20
    )

## Model

In [ ]:

df_model = aggregate_scores(df_valid_calls, ['model_short_name'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_model.pdf')
    print(fn)
        
    plot_grouped_metrics_labeled(
        df_model,
        group_configs=[
            {'label': 'Model', 
             'column': 'model_short_name', 
             'color': "#ABA6A3",'order': MODELS,
             'shade': False}
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * (len(MODELS)/5)),
        textwrap_width = 25,
    )

### Model and Language

In [ ]:
df_model_lang = aggregate_scores(df_valid_calls, ['model_short_name', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_model_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_model_lang,
        group_configs=[
            {'label': model, 
             'column': 'language_en', 
             'color': "#4F62C5",
            'filter': {'model_short_name': model}, 
            'order': LANGUAGE_ORDER}
            for model in df_model_lang.model_short_name.unique()
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * len(MODELS)/1.5),
        textwrap_width = 30
    )

## Model Family

In [ ]:
df_model_family = aggregate_scores(df_valid_calls, ['model_family'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_modelfamily.pdf')
    print(fn)
        
    plot_grouped_metrics_labeled(
        df_model_family,
        group_configs=[
            {'label': 'Model', 
             'column': 'model_family', 
             'color': "#D8B6DC",
             'order': MODEL_FAMILY_ORDER,
              'shade': False
             },
        ],
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * (len(df_model_family.model_family.unique())/6)),
        textwrap_width = 10,
        row_label_align='left',
        row_label_width=0.7,
        row_label_pad=0.22,
    )

### Model family and Language

In [ ]:
df_modelfamily_lang = aggregate_scores(df_valid_calls, ['model_family', 'language_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_modelfamily_lang.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_modelfamily_lang,
        group_configs=[
            {'label': model_family, 
             'column': 'language_en', 
             'color': "#4F62C5",
            'filter': {'model_family': model_family}, 
            'order': LANGUAGE_ORDER}
            for model_family in df_modelfamily_lang.model_family.unique()
        ],
        row_labels=LANGUAGE_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * df_modelfamily_lang.model_family.nunique()/2),
        textwrap_width = 12
    )

### Model familty and Country

In [ ]:
df_modelfamily_lang_loc = aggregate_scores(df_valid_calls, ['model_family', 'location_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_modelfamily_loc.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_modelfamily_lang_loc,
        group_configs=[
            {'label': model_family, 
             'column': 'location_en', 
             'color': "#4F62C5",
            'filter': {'model_family': model_family}, 
            'order': LOCATION_ORDER}
            for model_family in df_modelfamily_lang.model_family.unique()
        ],
        row_labels=LOCATION_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * df_modelfamily_lang.model_family.nunique()),
        textwrap_width = 12
    )

### Model family and Seniority

In [ ]:
df_modelfamily_lang_target = aggregate_scores(df_valid_calls, ['model_family', 'target_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_modelfamily_target.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_modelfamily_lang_target,
        group_configs=[
            {'label': model_family, 
             'column': 'target_en', 
             'color': "#4F62C5",
            'filter': {'model_family': model_family}, 
            'order': SENIORITY_ORDER}
            for model_family in df_modelfamily_lang.model_family.unique()
        ],
        row_labels=SENIORITY_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * df_modelfamily_lang.model_family.nunique()/2),
        textwrap_width = 12
    )

### Model familty and Field

In [ ]:
df_modelfamily_lang_field = aggregate_scores(df_valid_calls, ['model_family', 'field_en'])

for metric_type, metric_list in METRIC_TYPES.items():
    
    print(metric_type)
    
    fn = str(PLOTS_DIR / f'plot_{metric_type}_modelfamily_field.pdf')
    print(fn)

    plot_grouped_metrics_labeled(
        df_modelfamily_lang_field,
        group_configs=[
            {'label': model_family, 
             'column': 'field_en', 
             'color': "#4F62C5",
            'filter': {'model_family': model_family}, 
            'order': FIELD_ORDER}
            for model_family in df_modelfamily_lang.model_family.unique()
        ],
        row_labels=FIELD_MAP,
        metrics=metric_list,
        metric_directions=METRIC_DIRECTIONS,
        metric_labels=PLOT_LABELS,
        save_path=fn,
        figsize=(FIG_WIDTH, FIG_HEIGHT * df_modelfamily_lang.model_family.nunique()),
        textwrap_width = 12
    )